# 08 · Comparación e Interpretación de Segmentos

> **Objetivo:** comparar K-Means vs DBSCAN y traducir los clusters en
> perfiles accionables para el equipo de marketing.

## ¿Qué vamos a hacer?

1. Cargar ambos modelos y aplicar al mismo dataset.
2. Comparar sus métricas y distribuciones.
3. Construir el **perfil de negocio** de cada cluster.
4. Asignar **nombres** y **acciones recomendadas**.
5. Guardar el resultado final.

## ¿Cuándo es K-Means o DBSCAN la mejor elección?

| Dimensión | K-Means | DBSCAN |
|---|---|---|
| Tienes una idea clara del número de segmentos | ✅ | ❌ |
| Necesitas detectar outliers como tales | ❌ | ✅ |
| Forma de clusters | Esférica | Arbitraria |
| Sensibilidad a outliers | Alta | Robusta |
| Fácil de explicar a stakeholders | ✅ | Medio |
| Velocidad en datasets grandes | ✅ | OK |
| Dimensionalidad alta | OK | Sufre |

> **Spoiler:** en la práctica, muchas empresas usan K-Means para el
> análisis principal y DBSCAN como detector de outliers en paralelo.


In [ ]:
# Permite importar el paquete src/ desde el notebook
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import json
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from src.config import (
    FEATURES_DATA_FILE,
    EXTENDED_NUMERIC_FEATURES,
    KMEANS_MODEL_FILE,
    DBSCAN_MODEL_FILE,
    SEGMENTS_FILE,
    SEGMENT_PROFILES_FILE,
)
from src.features.preprocessing import build_preprocessing_pipeline
from src.models.clustering import evaluate_clustering
from src.models.profiling import build_segment_profiles, label_segments
from src.visualization.plots import plot_segment_radar


## 1. Cargar features y modelos

In [ ]:
features = pd.read_parquet(FEATURES_DATA_FILE)
pipeline = build_preprocessing_pipeline(numeric_features=EXTENDED_NUMERIC_FEATURES, use_log=True)
X = pipeline.fit_transform(features[EXTENDED_NUMERIC_FEATURES])

kmeans = joblib.load(KMEANS_MODEL_FILE)
dbscan = joblib.load(DBSCAN_MODEL_FILE)

features["cluster_kmeans"] = kmeans.labels_
# DBSCAN no expone .predict; reusamos labels_ porque entrenamos sobre el mismo X
features["cluster_dbscan"] = dbscan.labels_


## 2. Comparar métricas

In [ ]:
metrics_km = evaluate_clustering(X, kmeans.labels_)
metrics_db = evaluate_clustering(X, dbscan.labels_)

comparison = pd.DataFrame({"K-Means": metrics_km, "DBSCAN": metrics_db}).T
comparison.round(3)


**Cómo leerla:**

- `silhouette` → más alto mejor.
- `davies_bouldin` → más bajo mejor.
- `calinski_harabasz` → más alto mejor.
- `n_noise` → solo aplica a DBSCAN.

> Frecuentemente K-Means gana en silhouette (clusters esféricos compactos
> tras el log+escalado), pero DBSCAN aporta el valor diferencial de
> aislar outliers.

## 3. Distribución de clientes por modelo


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
features["cluster_kmeans"].value_counts().sort_index().plot.bar(
    ax=axes[0], color="steelblue"
)
axes[0].set_title("K-Means · clientes por cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Nº clientes")

features["cluster_dbscan"].value_counts().sort_index().plot.bar(
    ax=axes[1], color="darkorange"
)
axes[1].set_title("DBSCAN · clientes por cluster (-1 = ruido)")
axes[1].set_xlabel("Cluster")
plt.tight_layout()
plt.show()


## 4. Perfilamiento del modelo principal (K-Means)

Calculamos el promedio de las features originales por cluster.


In [ ]:
profiles_km = build_segment_profiles(
    features[EXTENDED_NUMERIC_FEATURES],
    labels=features["cluster_kmeans"].values,
)
profiles_km


## 5. Asignar nombres de negocio y acciones recomendadas

`label_segments` aplica una heurística basada en cuartiles relativos
de Recency, Frequency y Monetary entre los clusters.


In [ ]:
profiles_km_labeled = label_segments(profiles_km)
profiles_km_labeled[["n_customers", "share", "Recency", "Frequency", "Monetary",
                     "segment_name", "action"]]


**Lectura típica de los segmentos:**

| Segmento | Característica | Acción |
|---|---|---|
| Champions | Recency baja, Frequency alta, Monetary alto | VIP, recompensas. |
| Leales | Frequency alta, Monetary medio/alto | Cross-sell, referidos. |
| Nuevos / Ocasionales | Recency baja, Frequency baja | Bienvenida, nurturing. |
| En Riesgo | Recency alta, Monetary alto histórico | Reactivación urgente. |
| Inactivos | Recency alta, Frequency y Monetary bajos | Última oferta o baja. |

## 6. Visualización tipo radar


In [ ]:
fig = plot_segment_radar(
    profiles_km_labeled,
    features=["Recency", "Frequency", "Monetary", "AvgTicket", "ProductDiversity"],
)
fig.show()


> Cada polígono es un segmento. Si dos polígonos son muy similares,
> probablemente puedas fusionar esos clusters sin perder accionabilidad.

## 7. Perfilamiento de DBSCAN (incluye outliers)


In [ ]:
profiles_db = build_segment_profiles(
    features[EXTENDED_NUMERIC_FEATURES],
    labels=features["cluster_dbscan"].values,
)
profiles_db_labeled = label_segments(profiles_db)
profiles_db_labeled[["n_customers", "share", "Recency", "Frequency", "Monetary",
                     "segment_name", "action"]]


> El cluster `-1` representa los **clientes atípicos** detectados por DBSCAN.
> En este dataset suelen ser mayoristas con compras grandes. Vale la pena
> manejarlos con un equipo de cuentas dedicado, no con campañas masivas.

## 8. Guardar resultados finales


In [ ]:
features.reset_index().to_parquet(SEGMENTS_FILE, index=False)

profiles_export = {
    "kmeans": profiles_km_labeled.reset_index().to_dict(orient="records"),
    "dbscan": profiles_db_labeled.reset_index().to_dict(orient="records"),
}
SEGMENT_PROFILES_FILE.write_text(json.dumps(profiles_export, indent=2, default=str))

print(f"Segmentos por cliente: {SEGMENTS_FILE}")
print(f"Perfiles JSON:        {SEGMENT_PROFILES_FILE}")


## 9. Validación final con el negocio

Antes de declarar "victoria", responde:

1. ¿Cada segmento tiene un volumen mínimo accionable (≥ ~100 clientes)?
2. ¿Los nombres de los segmentos resuenan con marketing?
3. ¿Las acciones recomendadas se pueden ejecutar con las herramientas
   de email/CRM disponibles?
4. ¿Hay un plan de medición (uplift en conversión, retención, AOV)?

Si la respuesta a cualquiera es "no", itera: ajusta `k`, redefine features,
o discútelo con stakeholders.

## Resumen

- **K-Means** es la opción default, fácil de explicar.
- **DBSCAN** complementa con detección de outliers.
- Los clusters numéricos no sirven solos: hay que **traducirlos**
  a perfiles y acciones.
- La validación con el negocio es tan importante como las métricas.

---

## Preguntas de Reflexión

1. Si dos modelos dan métricas muy parecidas, ¿cómo decides cuál usar?
2. ¿Qué experimento harías para *medir* el impacto de la segmentación
   en el negocio?
3. ¿Qué riesgos éticos hay en segmentar clientes (sesgos, exclusión)?
4. Si dentro de un año el comportamiento de los clientes cambia, ¿cuándo
   y cómo re-entrenarías el modelo?

> **Cierre:** ahora abre la app de Streamlit (`app/streamlit_app.py`)
> y juega con los hiperparámetros para ver en vivo cómo cambian los
> segmentos.
